# 实践 5.1  从混池读段计数定位模拟QTL

《统计遗传与智能育种学》第五章 · 电子资源 `05_bsa_poolseq.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：05_bsa_poolseq.ipynb。实践模拟一条染色体上的SNP及两个极端池的测序读段，在已知目标区域制造等位基因频率分离。由参考/替代读段计数计算SNP-index、ΔSNP-index，并按双等位频率向量关系计算ED=√2|Δ|及ED⁴；随后使用不重叠分箱均值比较不同箱宽对峰位置误差的影响。ED⁴放大较大的频率差，但本身不自动给出统计显著性阈值。

输入：模拟SNP位置、两池目标等位基因频率与读段深度  
输出：delta、ED4，以及峰位置与真实QTL中心偏差。

## 输入、完整代码、实际输出与解释

输入与可复现条件：一条模拟染色体、两个极端池的读段计数和预设QTL区域；SEED=501；无需外部文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=501
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 501


In [2]:
n=900; pos=np.arange(n)*10000; qtl=4_500_000; base=rng.uniform(.2,.8,n); signal=.28*np.exp(-((pos-qtl)/650000)**2); p_hi=np.clip(base+signal,0.02,.98); p_lo=np.clip(base-signal,0.02,.98); depth_hi=rng.poisson(55,n)+1; depth_lo=rng.poisson(55,n)+1; alt_hi=rng.binomial(depth_hi,p_hi); alt_lo=rng.binomial(depth_lo,p_lo); idx_hi=alt_hi/depth_hi; idx_lo=alt_lo/depth_lo; delta=idx_hi-idx_lo; ed=np.sqrt(2.0)*np.abs(delta); ed4=ed**4
df=pd.DataFrame({'pos':pos,'depth_hi':depth_hi,'depth_lo':depth_lo,'SNP_index_hi':idx_hi,'SNP_index_lo':idx_lo,'delta':delta,'ED4':ed4})
print(df.head().round(4).to_string(index=False))

for w in [100000,300000,800000]:
 bins=np.arange(0,pos.max()+w,w); tmp=df.groupby(pd.cut(df.pos,bins,include_lowest=True),observed=True).agg(pos=('pos','mean'),delta=('delta','mean'),ED4=('ED4','mean')).dropna(); peak=float(tmp.loc[tmp.delta.abs().idxmax(),'pos']); print('window',w,'peak',int(peak),'localization error bp',int(abs(peak-qtl)))


  pos  depth_hi  depth_lo  SNP_index_hi  SNP_index_lo   delta    ED4
    0        57        53        0.6491        0.8868 -0.2377 0.0128
10000        58        56        0.5517        0.5000  0.0517 0.0000
20000        57        48        0.3509        0.5833 -0.2325 0.0117
30000        62        66        0.5161        0.4545  0.0616 0.0001
40000        62        51        0.7097        0.6863  0.0234 0.0000
window 100000 peak 4555000 localization error bp 55000
window 300000 peak 4655000 localization error bp 155000
window 800000 peak 4405000 localization error bp 95000


结果解释：代码从读段计数计算SNP-index、ΔSNP-index与ED⁴，并比较不重叠分箱。固定模拟中100 kb分箱峰距真值55 kb，300 kb和800 kb的峰位误差分别155 kb和95 kb，显示更宽分箱并不保证峰位单调改善；ED⁴只是放大较大的频率差，统计阈值仍需零模型或置换支持。

可修改练习：把平均深度减半并重复多个SEED，比较三种分箱宽度的峰位误差分布，而不是只比较一次运行的最高峰。